In [ ]:
# =========================================================
# Risk Adjusted Portfolio Metrics
# Beta, Jensen Alpha, Treynor Ratio
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

data_path = Path("data")
charts_path = Path("charts")
charts_path.mkdir(exist_ok=True)

# ---------------------------------------------------------
# Load Portfolio Data
# ---------------------------------------------------------

returns = pd.read_csv("india_equity_returns.csv", index_col="Date", parse_dates=True)
weights = pd.read_csv("portfolio_weights.csv", index_col=0)

# ---------------------------------------------------------
# Load NIFTY Data
# ---------------------------------------------------------

nifty = pd.read_csv("data/Nifty 50 Historical Data.csv")

# clean columns
nifty.columns = [c.strip() for c in nifty.columns]

# convert date
nifty["Date"] = pd.to_datetime(nifty["Date"])
nifty = nifty.sort_values("Date")
nifty = nifty.set_index("Date")

# convert price column
nifty["Price"] = nifty["Price"].astype(str).str.replace(",", "").astype(float)

# compute returns
nifty_returns = nifty["Price"].pct_change().dropna()

# ---------------------------------------------------------
# Align dates
# ---------------------------------------------------------

combined = pd.concat([returns, nifty_returns], axis=1).dropna()
combined.rename(columns={"Price":"NIFTY"}, inplace=True)

returns = combined.drop(columns=["NIFTY"])
nifty_returns = combined["NIFTY"]

# ---------------------------------------------------------
# Portfolio Returns
# ---------------------------------------------------------

equal_w = weights["Equal_Weight"].values
opt_w = weights["Optimized_Weight"].values

port_equal = returns @ equal_w
port_opt = returns @ opt_w

# ---------------------------------------------------------
# Risk Free Rate
# ---------------------------------------------------------

rf_annual = 0.05
rf_daily = rf_annual / 252

# ---------------------------------------------------------
# Beta Calculation
# ---------------------------------------------------------

def beta(port, market):

    cov = np.cov(port, market)[0,1]
    var = np.var(market)

    return cov / var

beta_equal = beta(port_equal, nifty_returns)
beta_opt = beta(port_opt, nifty_returns)

# ---------------------------------------------------------
# Jensen Alpha
# ---------------------------------------------------------

def jensen_alpha(port, market, beta):

    port_mean = port.mean()
    market_mean = market.mean()

    expected = rf_daily + beta * (market_mean - rf_daily)

    return port_mean - expected

alpha_equal = jensen_alpha(port_equal, nifty_returns, beta_equal)
alpha_opt = jensen_alpha(port_opt, nifty_returns, beta_opt)

# ---------------------------------------------------------
# Treynor Ratio
# ---------------------------------------------------------

def treynor(port, beta):

    port_mean = port.mean()

    return (port_mean - rf_daily) / beta

treynor_equal = treynor(port_equal, beta_equal)
treynor_opt = treynor(port_opt, beta_opt)

# ---------------------------------------------------------
# Compile Metrics Table
# ---------------------------------------------------------

metrics = pd.DataFrame({

    "Portfolio":[
        "Equal Weight",
        "Optimized"
    ],

    "Beta":[
        beta_equal,
        beta_opt
    ],

    "Jensen Alpha":[
        alpha_equal,
        alpha_opt
    ],

    "Treynor Ratio":[
        treynor_equal,
        treynor_opt
    ]

})

print(metrics)

# ---------------------------------------------------------
# Save Results
# ---------------------------------------------------------

metrics.to_csv("risk_adjusted_metrics.csv", index=False)

print("\nSaved: risk_adjusted_metrics.csv")